In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps as cm
from textwrap import fill

from internalizer.regionalization import regionalize_SE_mapping

In [ ]:
impact2color = {
    "acidification": cm["tab20"].colors[0],
    "climate change": cm["tab20"].colors[2],
    "ecotoxicity": cm["tab20"].colors[4],
    "eutrophication": cm["tab20"].colors[18],
    "fossil resources": cm["tab20"].colors[6],
    "human toxicity": cm["tab20"].colors[8],
    "ionizing radiation": cm["tab20"].colors[12],
    "land use": cm["tab20"].colors[16],
    "metal/mineral resources": cm["tab20"].colors[14],
    "ozone depletion": cm["tab20"].colors[3],
    "particulate matter formation": cm["tab20"].colors[10],
    "photochemical oxidant formation": cm["tab20"].colors[17],
    "water use": cm["tab20"].colors[1]
}

In [ ]:
datadir = "./lca/remind/SSP2-NPi-internalize-test-iterative-w-FE"
level = "FE"
year = 2050

df = pd.read_csv(datadir + f"/{year}/regionalized_costs_{level}.csv")
grouped = df.groupby(["dataset name", "impact category"])["cost"].mean().reset_index()
pdata = grouped.pivot(index="dataset name", columns="impact category", values="cost")

plt.figure(figsize=(15, 7))
pdata.plot.bar(stacked=True, color=impact2color)

In [ ]:
total = grouped.groupby("dataset name")["cost"].sum().sort_values(ascending=False)

techs_neg = total[(total < 0)].index

In [ ]:
sel = pdata.loc[techs_neg]

sel.plot.bar(stacked=True, color=impact2color)

In [ ]:
datadir = "./lca_coke-correction/remind/SSP2-NPi-internalize-test-iterative-w-FE"
level = "SE"
year = 2050

df = pd.read_csv(datadir + f"/{year}/regionalized_costs_{level}.csv")
grouped = df.groupby(["dataset name", "impact category"])["cost"].mean().reset_index()
pdata = grouped.pivot(index="dataset name", columns="impact category", values="cost")

after_correction = pdata.loc["coke production"]

plt.figure(figsize=(15, 7))
pdata.plot.bar(stacked=True, color=impact2color)

In [ ]:
datadir = "./lca_coke-correction/remind/SSP2-NPi-internalize-test-iterative-w-FE"
level = "FE"
year = 2050

df = pd.read_csv(datadir + f"/{year}/regionalized_costs_{level}.csv")
grouped = df.groupby(["dataset name", "impact category"])["cost"].mean().reset_index()
pdata = grouped.pivot(index="dataset name", columns="impact category", values="cost")

plt.figure(figsize=(15, 7))
pdata.plot.bar(stacked=True, color=impact2color)

In [ ]:
df

In [ ]:
def plot_costs_per_remind_index(datadir, year, level, sharex=True, xlim=None):
    # load cost data
    folder = datadir.split("/")[1]
    df = pd.read_csv(datadir + f"/{year}/regionalized_costs_{level}.csv").set_index(
        ["dataset name", "dataset reference product"]
    )

    # load mapping
    lvl2mapping = {
        "SE": "/p/tmp/davidba/internalizer/internalizer/data/mappings/prodSE.csv",
        "FE": "/p/tmp/davidba/internalizer/internalizer/data/mappings/demFE.csv",
        "fe": "/p/tmp/davidba/internalizer/internalizer/data/mappings/demFE.csv",
        "pe2se": "/p/tmp/davidba/internalizer/internalizer/data/mappings/pe2se.csv",
        "se2h2": "/p/tmp/davidba/internalizer/internalizer/data/mappings/se2h2.csv",
        "h22se": "/p/tmp/davidba/internalizer/internalizer/data/mappings/h22se.csv",
    }
    mapping = pd.read_csv(lvl2mapping[level], sep=";")

    unique_indices = list(mapping["REMIND index"].unique())
    N = len(unique_indices)

    # get techs per REMIND index
    techs = []
    for idx in unique_indices:
        sel = mapping.loc[mapping["REMIND index"] == idx].copy()
        tech_idx = pd.MultiIndex.from_frame(sel[["dataset name", "dataset reference product"]])
        techs.append(tech_idx.drop_duplicates())

    fig, axs = plt.subplots(N, 1, figsize=(10, 1+3*N), sharex=sharex,
                            height_ratios=[1+len(tidx) for tidx in techs])
    if N == 1:
        all_axes = [axs]
    else:
        all_axes = axs.flat
    for ax, idx, tech_idx in zip(all_axes, unique_indices, techs):
        ax.set_title(idx)
        data_sel = df.loc[tech_idx].copy().reset_index()
        summed_output = data_sel.groupby(["dataset name", "impact category", "region"])["cost"].sum().reset_index()
        grouped = summed_output.groupby(["dataset name", "impact category"])["cost"].mean().reset_index()
        pdata = grouped.pivot(index="dataset name", columns="impact category", values="cost")

        labels = [fill(s, 40) for s in pdata.index]

        ax.axvline(x=0, color="gray", lw=0.8)
        pdata.plot.barh(stacked=True, ax=ax, color=impact2color).legend(loc="upper right", fontsize=7)
        ax.set_yticklabels(labels)

        if xlim is not None:
            ax.set_xlim(xlim)

    fig.tight_layout()
    pdfname = f"./cost_checks/{folder}_{level}_{year}.pdf"
    if sharex:
        pdfname = f"./cost_checks/{folder}_{level}_{year}_sharex.pdf"
    fig.savefig(pdfname)
    plt.close()




In [ ]:
xlims_new = {
    "pe2se": [-0.04, 0.30],
    "se2h2": [-0.04, 0.30],
    "h22se": [-0.04, 0.30],
    "fe": [-0.02, 0.10]
}
for level in ["pe2se", "se2h2", "h22se", "fe"]:
    for folder in ["lca_v2"]:
        datadir = f"./{folder}/remind/SSP2-NPi-internalize-combined"
        plot_costs_per_remind_index(datadir, 2050, level, sharex=False, xlim=xlims_new[level])